# 🎯 FPGA Radar UAV Classifier — Model Training
## Defense Systems • BITS Pilani AMD/Xilinx FPGA Hackathon 2026

**Objective:** Train a lightweight RadarCNN on Range-Doppler maps for real-time UAV/target classification.

| Spec | Value |
|------|-------|
| Input | 1×11×61 Range-Doppler map |
| Architecture | Conv2d(1→8→16) + MaxPool + FC(480→32→3) |
| Classes | drone (0), car (1), person (2) |
| Target | >92% test accuracy |
| Export | model.pth for hls4ml quantization |

---
## 1. Imports & Setup

In [ ]:
import os
import sys
import time
import json

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Configuration
DATASET_DIR  = os.path.join("..", "01_Dataset")
MAPS_PATH    = os.path.join(DATASET_DIR, "rd_maps.npy")
LABELS_PATH  = os.path.join(DATASET_DIR, "labels.npy")
CLASS_NAMES  = ["drone", "car", "person"]
NUM_CLASSES  = 3
INPUT_H, INPUT_W = 11, 61

# Hyperparameters
EPOCHS      = 20
BATCH_SIZE  = 64
LR          = 1e-3
WEIGHT_DECAY = 1e-4
SEED        = 42

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")

---
## 2. Load & Preprocess Dataset

In [ ]:
# Load NumPy arrays from dataset loader
rd_maps = np.load(MAPS_PATH)    # (N, 11, 61)
labels  = np.load(LABELS_PATH)  # (N,)

print(f"Dataset: {rd_maps.shape[0]:,} samples")
print(f"Shape:   {rd_maps.shape}")
print(f"Labels:  {np.unique(labels)} → {CLASS_NAMES}")
print(f"Value range: [{rd_maps.min():.2f}, {rd_maps.max():.2f}] dBm")

# Class distribution
for i, name in enumerate(CLASS_NAMES):
    count = (labels == i).sum()
    print(f"  {name:>6s}: {count:>5,} ({100*count/len(labels):.1f}%)")

In [ ]:
# Visualize sample Range-Doppler maps (one per class)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, name in enumerate(CLASS_NAMES):
    idx = np.where(labels == i)[0][0]
    im = axes[i].imshow(rd_maps[idx], aspect='auto', cmap='viridis', origin='lower')
    axes[i].set_title(f'{name.upper()} (label={i})', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Doppler Bin')
    axes[i].set_ylabel('Range Bin')
    plt.colorbar(im, ax=axes[i], label='dBm')
plt.suptitle('Sample Range-Doppler Maps', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Normalize: per-sample zero-mean, unit-variance
mean = rd_maps.mean(axis=(1, 2), keepdims=True)
std  = rd_maps.std(axis=(1, 2), keepdims=True) + 1e-8
rd_maps_norm = ((rd_maps - mean) / std).astype(np.float32)

# Add channel dim: (N, 11, 61) → (N, 1, 11, 61)
rd_maps_norm = rd_maps_norm[:, np.newaxis, :, :]

# Train / Val / Test split (70/10/20)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    rd_maps_norm, labels, test_size=0.2, random_state=SEED, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.125, random_state=SEED, stratify=y_trainval
)

print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")

# PyTorch DataLoaders
def make_loader(X, y, shuffle=False):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val, y_val)
test_loader  = make_loader(X_test, y_test)

---
## 3. RadarCNN Architecture
Designed for FPGA deployment — small filter counts, standard ops supported by hls4ml.

In [ ]:
class RadarCNN(nn.Module):
    """
    Lightweight CNN for Range-Doppler classification.
    
    Architecture:
      Conv2d(1→8, 3×3) → BN → ReLU → MaxPool(2×2)     → 8×5×30
      Conv2d(8→16, 3×3) → BN → ReLU → MaxPool(2×2)    → 16×2×15
      Flatten → FC(480→32) → ReLU → Dropout → FC(32→3)
    """
    def __init__(self, num_classes=3):
        super().__init__()
        # Block 1
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(8)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Block 2
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(16)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Classifier
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(16 * 2 * 15, 32)
        self.relu3   = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.3)
        self.fc2     = nn.Linear(32, num_classes)
    
    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.flatten(x)
        x = self.dropout(self.relu3(self.fc1(x)))
        return self.fc2(x)


model = RadarCNN().to(device)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Float32 size:     {total_params * 4 / 1024:.1f} KB")
print(f"Int8 size (est.): {total_params * 1 / 1024:.1f} KB")

# Verify forward pass shape
dummy = torch.randn(1, 1, INPUT_H, INPUT_W).to(device)
out = model(dummy)
print(f"\nInput:  {dummy.shape} → Output: {out.shape}")

---
## 4. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_epoch = 0

for epoch in tqdm(range(1, EPOCHS + 1), desc="Training"):
    # ── Train ────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        train_correct += outputs.argmax(1).eq(targets).sum().item()
        train_total += targets.size(0)
    
    # ── Validate ─────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            val_correct += outputs.argmax(1).eq(targets).sum().item()
            val_total += targets.size(0)
    
    scheduler.step()
    
    # Record
    t_loss = train_loss / train_total
    t_acc  = 100 * train_correct / train_total
    v_loss = val_loss / val_total
    v_acc  = 100 * val_correct / val_total
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    
    mark = '★' if v_acc > best_val_acc else ' '
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch
        torch.save(model.state_dict(), 'model.pth')
        torch.save(model, 'model_full.pth')
    
    print(f"  Epoch {epoch:2d}/{EPOCHS} │ Train {t_acc:6.2f}% │ Val {v_acc:6.2f}% │ {mark}")

print(f"\n✅ Best val accuracy: {best_val_acc:.2f}% (epoch {best_epoch})")

---
## 5. Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], 'b-o', ms=3, label='Train')
ax1.plot(epochs_range, history['val_loss'],   'r-s', ms=3, label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'b-o', ms=3, label='Train')
ax2.plot(epochs_range, history['val_acc'],   'r-s', ms=3, label='Val')
ax2.axhline(92, color='green', ls='--', alpha=0.7, label='92% Target')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curves'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plots/training_curves.png', dpi=150)
plt.show()

---
## 6. Test Set Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('model.pth', map_location=device, weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs.to(device))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(targets.numpy())

y_pred = np.array(all_preds)
y_true = np.array(all_labels)
test_acc = 100 * (y_pred == y_true).mean()

print(f"Test Accuracy: {test_acc:.2f}%")
print(f"{'✅ TARGET MET' if test_acc >= 92 else '⚠️  Below target'}")
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=range(3), yticks=range(3),
       xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
       xlabel='Predicted', ylabel='True',
       title='Confusion Matrix — Test Set')

thresh = cm.max() / 2
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > thresh else 'black',
                fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/confusion_matrix.png', dpi=150)
plt.show()

---
## 7. Model Summary for FPGA Export
Information needed for hls4ml conversion in the next stage.

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print("=" * 50)
print("MODEL READY FOR FPGA DEPLOYMENT")
print("=" * 50)
print(f"Parameters:     {total_params:,}")
print(f"Input shape:    (1, 1, {INPUT_H}, {INPUT_W})")
print(f"Output classes: {NUM_CLASSES} — {CLASS_NAMES}")
print(f"Test accuracy:  {test_acc:.2f}%")
print(f"Model file:     model.pth")
print(f"Target FPGA:    xc7z020clg400-1 (ZedBoard/PYNQ)")
print(f"Float32 size:   {total_params * 4 / 1024:.1f} KB")
print(f"Int8 size:      {total_params / 1024:.1f} KB")
print("=" * 50)
print("\n✅ Ready for hls4ml quantization & Verilog generation!")